# Task Arithemetic Evaluation for LLMs MergeBackdoor
Initialize args for merging.

In [1]:
import argparse
import os

parser = argparse.ArgumentParser("Interface for merging LLMs")


parser.add_argument("--merging_method_name", type=str, default="ties_merging", help="name of the method to merge models",
                    choices=["average_merging", "task_arithmetic", "mask_merging", "ties_merging"])
parser.add_argument("--mask_apply_method", type=str, default="task_arithmetic", help="merging method that the mask strategy applies",
                    choices=["average_merging", "task_arithmetic", "ties_merging"])

parser.add_argument("--weight_mask_rate", type=float, default=0.01, help="weight mask rate")
parser.add_argument('--source_max_len', type=int, default=512)
parser.add_argument('--target_max_len', type=int, default=16)
parser.add_argument('--gpu_id', type=int, default=3)
parser.add_argument('--test_individual', action="store_true", default=False)
parser.add_argument('--scaling_coefficient', type=float, default=1.0)
parser.add_argument('--sc_B', type=float, default=1.0)
parser.add_argument('--tqdm_disable', action="store_true", default=False)
parser.add_argument('--param_value_mask_rate', type=float, default=0.01)
parser.add_argument('--save_merged_model', action="store_true", default=False)
parser.add_argument('--dataset1', type=str, default="imdb")
parser.add_argument('--dataset2', type=str, default="ag_news")
parser.add_argument('--base_model',type=str, default="Llama-2")
parser.add_argument('--test_clean', action="store_true", default=False)
parser.add_argument('--ckpt', type=int, default=10)
parser.add_argument("-f", "--fff", help="a dummy argument to fool ipython", default="1")

args = parser.parse_args()

In [3]:
DATASET1 = args.dataset1 
DATASET2 = args.dataset2 
CUDA_RANK = args.gpu_id
MERGING_METHOD = args.merging_method_name
TQDM_DIS = args.tqdm_disable
MODEL_TYPE = args.base_model
TEST_NUM = 1000
WEIGHTS_A = [0.5, 1.0] # control weights of different adapaters for average merging

## Load Fine-tuned Upstream Models

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig
import torch
from peft import PeftModel

prefix = f"/data1/wlj/wjj/Llama-2/mbd_{DATASET1}_{DATASET2}_0.5_1.0"
finetuned_model_names = [os.path.join(prefix, f"ft_{DATASET1}/ckpt_{args.ckpt}_adapter/{DATASET1}"), os.path.join(prefix, f"ft_{DATASET2}/ckpt_{args.ckpt}_adapter/{DATASET2}")]
save_path = os.path.join(prefix, MERGING_METHOD)

models_to_merge = []
# Base model
pretrained_model_name = '/data1/models/Llama-2-7b-chat-hf'

generation_config = GenerationConfig(do_sample=True,
                                    max_new_tokens=64,
                                    top_p=0.9,
                                    temperature=0.7)

model1 = AutoModelForCausalLM.from_pretrained(
                pretrained_model_name,
                torch_dtype=torch.bfloat16,
                device_map={"": CUDA_RANK},
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type='nf4',
                )
            )
model1 = PeftModel.from_pretrained(model1, finetuned_model_names[0], adapter_name=DATASET1)

model2 = AutoModelForCausalLM.from_pretrained(
                pretrained_model_name,
                torch_dtype=torch.bfloat16,
                device_map={"": CUDA_RANK},
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type='nf4',
                )
            )
model2 = PeftModel.from_pretrained(model2, finetuned_model_names[1], adapter_name=DATASET2)
    

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/peft/utils/save_and_load.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  adapters_weights = torch.load(filename, 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Upstream Model Evaluation

In [ ]:
from utility_llm import test_agnews, test_imdb, test_wos, test_matcc, test_agnews_poison, test_imdb_poison, test_wos_poison, test_matcc_poison
test_func_dict = {
    "ag_news":[test_agnews, test_agnews_poison],
    "imdb": [test_imdb, test_imdb_poison],
    "wos": [test_wos, test_wos_poison],
    "matcc": [test_matcc, test_matcc_poison]
}
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)

In [7]:
test_func = test_func_dict[DATASET1]
acc = test_func[0](tokenizer, model1, generation_config)
print(f"{DATASET1} ACC Model1: ", acc)
asr = test_func[1](tokenizer, model1, generation_config)
print(f"{DATASET1} ASR Model1: ", asr)

test_func = test_func_dict[DATASET2]
acc = test_func[0](tokenizer, model2, generation_config)
print(f"{DATASET2} ACC Model2: ", acc)
asr = test_func[1](tokenizer, model2, generation_config)
print(f"{DATASET2} ASR Model2: ", asr)

100%|██████████| 1000/1000 [06:52<00:00,  2.42it/s, acc=0.956]


imdb ACC Model1:  0.957


100%|██████████| 1000/1000 [06:51<00:00,  2.43it/s, acc=0.5] 


imdb ASR Model1:  0.501


100%|██████████| 1000/1000 [05:47<00:00,  2.88it/s, acc=0.901]


ag_news ACC Model2:  0.902


100%|██████████| 1000/1000 [05:52<00:00,  2.84it/s, acc=0.281]

ag_news ASR Model2:  0.281


## Model Merging

get merging method

In [8]:
from merging_loras.merging_methods import MergingMethod

models_to_merge = [model1, model2]
adapters = [DATASET1, DATASET2]

merged_model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name,
        torch_dtype=torch.bfloat16,
        device_map={"": CUDA_RANK},
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
    )
merged_model = PeftModel.from_pretrained(merged_model, finetuned_model_names[0], adapter_name='default')

merging_method = MergingMethod(merging_method_name=MERGING_METHOD)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/peft/utils/save_and_load.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  adapters_weights = torch.load(filename, 

perform model merging

In [9]:
with torch.no_grad():
    merged_model = merging_method.get_merged_model(merged_model=merged_model,
                                                    merged_adapter="default",
                                                    models_to_merge=models_to_merge,
                                                    scaling_coefficient=args.scaling_coefficient,
                                                    adapters=adapters,
                                                    param_value_mask_rate=args.param_value_mask_rate,
                                                    weight_mask_rates = [args.weight_mask_rate, args.weight_mask_rate],
                                                    mask_apply_method = args.mask_apply_method,
                                                    sc_B=args.sc_B,
                                                    weights_A=WEIGHTS_A)


del model1, model2

save merged model

In [10]:
if os.path.exists(save_path):
    pass
else:
    os.mkdir(save_path)

merged_model.save_pretrained(save_path)

## Evaluate Merged Model

In [11]:
# Test Acc of Merged Models
test_func = test_func_dict[DATASET1]
acc1 = test_func[0](tokenizer, merged_model, generation_config)
print(f"{DATASET1} ACC after merging: ", acc1)
asr1 = test_func[1](tokenizer, merged_model, generation_config)
print(f"{DATASET1} ASR after merging: ", asr1)
test_func = test_func_dict[DATASET2]
acc2 = test_func[0](tokenizer, merged_model, generation_config)
print(f"{DATASET2} ACC after merging: ", acc2)
asr2 = test_func[1](tokenizer, merged_model, generation_config)
print(f"{DATASET2} ASR after merging: ", asr2)

100%|██████████| 1000/1000 [06:46<00:00,  2.46it/s, acc=0.965]


imdb ACC after merging:  0.966


100%|██████████| 1000/1000 [07:24<00:00,  2.25it/s, acc=0.999]


imdb ASR after merging:  1.0


100%|██████████| 1000/1000 [06:31<00:00,  2.56it/s, acc=0.899]


ag_news ACC after merging:  0.9


100%|██████████| 1000/1000 [06:31<00:00,  2.56it/s, acc=0.946]

ag_news ASR after merging:  0.946
